##### Notebook responsável por realizar a extração do dolar, ipca, selic e brent(Petróleo Bruto Brent - Dados Financeiros) e armazenar num arquivo .parquet

### 1. Importações

In [1]:
import pandas as pd
import requests
import yfinance as yf
from datetime import datetime, timedelta
import holidays

#f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial={date_init}&@dataFinalCotacao={date_final}&$top=10000&$format=json&$select=cotacaoCompra,dataHoraCotacao"

### 2. Creating DataFrame dolar

In [2]:
# #Dont using 
# actual_date = datetime.today().strftime('%d/%m/%Y')

# datas = [
#     "10/05/2004",
#     "10/05/2014",
#     "10/05/2024",
#     actual_date
# ]
# df = []
# # Loop de 2 em 2
# for i in range(0, len(datas), 2):
#     d1 = datas[i]
#     d2 = datas[i+1]
#     print(d1,d2)
#     data = requests.get(f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.1/dados?formato=json&dataInicial={d1}&dataFinal={d2}")
#     df.append(data.json())

#### 2.1 Applying data in archive .parquet

In [3]:
# doc api: https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/aplicacao#!/recursos/CotacaoDolarPeriodo#eyJmb3JtdWxhcmlvIjp7IiRmb3JtYXQiOiJqc29uIiwiJHRvcCI6MTAwMDAsImRhdGFJbmljaWFsIjoiMDEtMDEtMjAwNCIsImRhdGFGaW5hbENvdGFjYW8iOiIwMS0wMS0yMDEzIn0sInByb3ByaWVkYWRlcyI6WzEsMl19
actual_date = datetime.today()#.strftime('%m-%d-%Y')
match actual_date.weekday():
    case 6: actual_date = actual_date - timedelta(days=2)
    case 5: actual_date = actual_date - timedelta(days=1)
actual_date = actual_date.strftime('%m-%d-%Y')
print(actual_date)


#filtering date in 10year in 10
datas = [
    '05-10-2004',
    '05-10-2014',
    '05-10-2024',
    actual_date
]
# purchase dolar
values_dolar_purchase = []
dates_dolar_purchase = []
# sale dolar
values_dolar_sale = []
dates_dolar_sale = []
for buy_sale in ["cotacaoCompra","cotacaoVenda"]:
    # Loop by 2 in 2 by date
    for i in range(0, len(datas), 2):
        date_init = datas[i]
        date_final = datas[i+1]
        print(date_init,date_final)  
        resp = requests.get(f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{date_init}'&@dataFinalCotacao='{date_final}'&$top=10000&$format=json&$select={buy_sale},dataHoraCotacao") # cotacaoCompra
        data = resp.json()
        match buy_sale:
            case "cotacaoCompra":
                [values_dolar_purchase.append(data["value"][i]["cotacaoCompra"]) for i in range(len(data["value"]))]
                [dates_dolar_purchase.append(data["value"][i]["dataHoraCotacao"]) for i in range(len(data["value"]))]
            case "cotacaoVenda":
                [values_dolar_sale.append(data["value"][i]["cotacaoVenda"]) for i in range(len(data["value"]))]
                [dates_dolar_sale.append(data["value"][i]["dataHoraCotacao"]) for i in range(len(data["value"]))]

08-12-2026
05-10-2004 05-10-2014
05-10-2024 08-12-2026
05-10-2004 05-10-2014
05-10-2024 08-12-2026


In [4]:
dolar = {"Valor Venda Dolar": values_dolar_sale, 
        "Valor Compra Dolar": values_dolar_purchase, 
        "Data Compra Venda Dolar": dates_dolar_purchase}
df_dolar = pd.DataFrame(dolar)
df_dolar["Data Compra Venda Dolar"] = pd.to_datetime(df_dolar["Data Compra Venda Dolar"]).dt.strftime('%Y-%m-%d')
df_dolar["Valor Venda Dolar"] = df_dolar["Valor Venda Dolar"].astype(float)
df_dolar["Valor Compra Dolar"] = df_dolar["Valor Compra Dolar"].astype(float)
# transform date in index and drop 
df_dolar.index = df_dolar["Data Compra Venda Dolar"]
df_dolar = df_dolar.drop(columns=["Data Compra Venda Dolar"])

In [5]:
df_dolar.to_parquet("..\data\dolar.parquet")#data turn index

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ferna\AppData\Local\Temp\ipykernel_3096\506106535.py:1: SyntaxWarning: invalid escape sequence '\d'
  df_dolar.to_parquet("..\data\dolar.parquet")#data turn index


### 3. Getting Brent and applying in the same archive .parquet dolar

In [6]:
# Define o ticker do Brent (BZ=F)
ticker = "BZ=F"

# Define as datas (do início de 2004 até a data de hoje)
data_inicio = "2004-01-01"
data_fim = datetime.today().strftime('%Y-%m-%d')

print(f"Baixando dados do Brent ({ticker}) de {data_inicio} até {data_fim}...")

# Baixa os dados
# auto_adjust=True ajuda a corrigir distorções de contratos antigos
dados = yf.download(ticker, start=data_inicio, end=data_fim, auto_adjust=True)

if not dados.empty:
    print(f"\nSucesso! Foram baixados {len(dados)} dias de negociação.")
    
    # Exibe os 5 primeiros registros (2004/2007)
    print("\n--- Primeiros Registros ---")
    print(dados.head())

    # Exibe os 5 últimos registros (Hoje)
    print("\n--- Últimos Registros ---")
    print(dados.tail())
    

else:
    print("Não foi possível encontrar dados para este período. O Yahoo pode ter mudado o ticker.")

Baixando dados do Brent (BZ=F) de 2004-01-01 até 2026-08-12...


[*********************100%***********************]  1 of 1 completed


Sucesso! Foram baixados 4738 dias de negociação.

--- Primeiros Registros ---
Price           Close       High        Low       Open Volume
Ticker           BZ=F       BZ=F       BZ=F       BZ=F   BZ=F
Date                                                         
2007-07-30  75.739998  76.529999  75.440002  75.849998   2575
2007-07-31  77.050003  77.169998  75.669998  75.699997   3513
2007-08-01  75.349998  77.059998  74.860001  77.000000   3930
2007-08-02  75.760002  76.209999  74.269997  75.220001   6180
2007-08-03  74.750000  76.000000  74.529999  75.389999   4387

--- Últimos Registros ---
Price           Close       High        Low       Open Volume
Ticker           BZ=F       BZ=F       BZ=F       BZ=F   BZ=F
Date                                                         
2026-08-06  82.489998  83.769997  78.980003  79.430000  40651
2026-08-07  83.550003  84.389999  81.500000  83.330002  37626
2026-08-10  87.720001  87.919998  83.339996  83.559998  30023
2026-08-11  88.910004  90.

## 4. Merge Dolar csv with ANP parquet

### 4.1 Validando dfs (dolar e brent)

tratamento feito, pois em dias de feriados BR não feito o lançamento do valor de venda/compra do dolar pela api

In [7]:
# tirando multindex do df brent(dados)
df_single_index = dados.copy()
df_single_index.columns = ['_'.join(col).strip() for col in dados.columns]
# aplicando o index do df_dolar em datetime, para realizar a validacao se há datas do df_brent(dados) que nao estao no df_dolar
df_dolar.index = pd.to_datetime(df_dolar.index)
df_single_index = df_single_index.reset_index()
df_single_index = df_single_index.set_index(["Date"])

In [8]:
# dados dolar que nao estao no brent -> sao feriados BR que nao foram liberados pela api do dolar
df_single_index[~df_single_index.index.isin(df_dolar.index)]

,Close_BZ=F,High_BZ=F,Low_BZ=F,Open_BZ=F,Volume_BZ=F
Date,,,,,
2007-09-07,75.070000,75.220001,74.279999,74.360001,3685
2007-10-12,80.550003,80.889999,79.790001,79.870003,1864
2007-11-02,92.080002,92.230003,90.260002,90.610001,1781
2007-11-15,90.230003,90.650002,88.720001,90.559998,935
2008-02-04,90.400002,91.110001,88.660004,89.010002,1760
...,...,...,...,...,...
2025-11-20,63.380001,64.389999,62.950001,63.590000,42345
2026-02-17,67.419998,69.029999,66.830002,68.059998,74745
2026-04-21,98.480003,101.150002,93.870003,93.989998,50039


### 4.2 Tratando df_dolar

In [9]:
# df_dolar = pd.read_csv(r"..\data\dolar.csv", sep=",", index_col=0)
# df = pd.read_parquet(r"..\data\anp_parquet.parquet")
# df["Data da Coleta"] = pd.to_datetime(df["Data da Coleta"])
# df = df.drop(columns=["Regiao - Sigla", "Nome da Rua","Numero Rua", "Complemento", "Bairro", "Regiao - Sigla.1"])
# # drop values nan from columns Bandeira and Unidade de Medida
# df = df.dropna(subset=["Bandeira","Unidade de Medida"])
# df["Unidade de Medida"] = df["Unidade de Medida"].replace("R$ / mÂ³","R$ / m³")


In [10]:
date_init = '2004-05-10'
date_final = '2026-05-09'
all_dates = pd.date_range(date_init, date_final, freq='D')
df_calendar = pd.DataFrame(all_dates, columns=['data'])


In [11]:
df_calendar["is_weekend"] = df_calendar["data"].dt.weekday >= 5
br_feriados = holidays.Brazil(years=range(2004, datetime.today().year))
# Verifica se cada data no DataFrame é um feriado
df_calendar['is_holiday'] = df_calendar['data'].isin(br_feriados)

# 5. Criar uma coluna que marca qualquer dia não útil
df_calendar['day_no_util'] = df_calendar['is_weekend'] | df_calendar['is_holiday']
dias_nao_uteis = df_calendar[df_calendar['day_no_util']].copy()
print(f"Total de dias não úteis no período: {len(dias_nao_uteis)}")
print(dias_nao_uteis.head(10))

Total de dias não úteis no período: 2441
         data  is_weekend  is_holiday  day_no_util
5  2004-05-15        True       False         True
6  2004-05-16        True       False         True
12 2004-05-22        True       False         True
13 2004-05-23        True       False         True
19 2004-05-29        True       False         True
20 2004-05-30        True       False         True
26 2004-06-05        True       False         True
27 2004-06-06        True       False         True
33 2004-06-12        True       False         True
34 2004-06-13        True       False         True


C:\Users\ferna\AppData\Local\Temp\ipykernel_3096\3176613655.py:4: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df_calendar['is_holiday'] = df_calendar['data'].isin(br_feriados)


In [12]:
df_calendar[df_calendar["is_holiday"] == True]#["data"]#[df_calendar["is_holiday"] == True]["data"].max()

,data,is_weekend,is_holiday,day_no_util
120,2004-09-07,False,True,True
155,2004-10-12,False,True,True
176,2004-11-02,False,True,True
189,2004-11-15,False,True,True
229,2004-12-25,True,True,True
...,...,...,...,...
7825,2025-10-12,True,True,True
7846,2025-11-02,True,True,True
7859,2025-11-15,True,True,True
7864,2025-11-20,False,True,True


In [13]:
df_dolar[df_dolar.index.duplicated() == True]

,Valor Venda Dolar,Valor Compra Dolar
Data Compra Venda Dolar,,
2025-04-23,5.688,5.6874


In [14]:

# Caso tenha valores nan, drop em 1 e mantenha o primeiro registro do index
df_dolar = df_dolar[~df_dolar.index.duplicated(keep='first')]


df_dolar.index = pd.to_datetime(df_dolar.index)

# Reindexar o DataFrame para incluir TODOS os dias do período
# Isso criará linhas com valores NaN para os dias que estavam faltando (não úteis)
todas_as_datas_serie = pd.date_range(start=df_dolar.index.min(), end=df_dolar.index.max(), freq='D')

dolar_completo = df_dolar.reindex(todas_as_datas_serie)

# Preencher os valores faltantes (os dias não úteis) com o último valor válido
# Forward fill - preenche NaN com último valor válido anterior
dolar_completo['Valor Venda Dolar'] = dolar_completo['Valor Venda Dolar'].ffill()
dolar_completo['Valor Compra Dolar'] = dolar_completo['Valor Compra Dolar'].ffill()

dolar_completo.index = dolar_completo.index.rename("Data Coleta Dolar")
# dolar_completo

In [15]:
# validando dnv
df_single_index[~df_single_index.index.isin(dolar_completo.index)]

,Close_BZ=F,High_BZ=F,Low_BZ=F,Open_BZ=F,Volume_BZ=F
Date,,,,,


## 5. Merge df_dolar + df_brent

In [16]:
df_single_index

,Close_BZ=F,High_BZ=F,Low_BZ=F,Open_BZ=F,Volume_BZ=F
Date,,,,,
2007-07-30,75.739998,76.529999,75.440002,75.849998,2575
2007-07-31,77.050003,77.169998,75.669998,75.699997,3513
2007-08-01,75.349998,77.059998,74.860001,77.000000,3930
2007-08-02,75.760002,76.209999,74.269997,75.220001,6180
2007-08-03,74.750000,76.000000,74.529999,75.389999,4387
...,...,...,...,...,...
2026-08-06,82.489998,83.769997,78.980003,79.430000,40651
2026-08-07,83.550003,84.389999,81.500000,83.330002,37626
2026-08-10,87.720001,87.919998,83.339996,83.559998,30023


In [17]:
# ocorrerá perda de dados do dolar no ano de 2004, mas bem nao irá causar problemas por conta que os dados utilizados 
# no modelo nao terao esse impacto
df_dolar_enriched = pd.merge(dolar_completo, df_single_index, 
    left_index=True,
    right_index=True,
    how="inner")

In [18]:
df_dolar_enriched.to_parquet(r"..\data\dolar_enriched.parquet")

# Extracting IPCA, SELIC & PIB_mensal

In [19]:
codes = {
    'ipca': 433,
    'selic': 11,
    'pib_mensal': 24363#24369,
}
actual_date = datetime.today().strftime('%d/%m/%Y')
dates = [
    '10/05/2004',
    '10/05/2014',
    '10/05/2024',
    actual_date
]
c = 0
data_collected = {
    433: [],
    11: [],
    24363: []
}
for i in range(0, len(dates), 2):
    for code in codes:
        url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codes[code]}/dados?formato=json&dataInicial={dates[i]}&dataFinal={dates[i+1]}"
        resp = requests.get(url)
        print(url)
        print(f"{codes[code]}:{resp.json()}")
        data_collected[codes[code]].append(resp.json())
        c += 1
        if c == 2:
            print("=============")
            c=0

https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=10/05/2004&dataFinal=10/05/2014
433:[{'data': '01/05/2004', 'valor': '0.51'}, {'data': '01/06/2004', 'valor': '0.71'}, {'data': '01/07/2004', 'valor': '0.91'}, {'data': '01/08/2004', 'valor': '0.69'}, {'data': '01/09/2004', 'valor': '0.33'}, {'data': '01/10/2004', 'valor': '0.44'}, {'data': '01/11/2004', 'valor': '0.69'}, {'data': '01/12/2004', 'valor': '0.86'}, {'data': '01/01/2005', 'valor': '0.58'}, {'data': '01/02/2005', 'valor': '0.59'}, {'data': '01/03/2005', 'valor': '0.61'}, {'data': '01/04/2005', 'valor': '0.87'}, {'data': '01/05/2005', 'valor': '0.49'}, {'data': '01/06/2005', 'valor': '-0.02'}, {'data': '01/07/2005', 'valor': '0.25'}, {'data': '01/08/2005', 'valor': '0.17'}, {'data': '01/09/2005', 'valor': '0.35'}, {'data': '01/10/2005', 'valor': '0.75'}, {'data': '01/11/2005', 'valor': '0.55'}, {'data': '01/12/2005', 'valor': '0.36'}, {'data': '01/01/2006', 'valor': '0.59'}, {'data': '01/02/2006

In [20]:
len_datas = 0
for i in data_collected:
    datas = data_collected[i]
    for n in range(len(datas)):
        data = datas[n]
        len_datas += len(data)
        for i2 in data:
            ...# print(f"{i}:{i2}")
print(len_datas)

3375


In [21]:

rows = []

for codigo in data_collected:
    listas = data_collected[codigo]     # lista de listas

    for lista in listas:                # lista interna
        for item in lista:              # dict final
            rows.append({
                "codigo": codigo,
                "data": item.get("data"),
                "valor": float(item.get("valor"))
            })

df = pd.DataFrame(rows)

In [22]:
df = df.replace({433:'ipca',11:'selic',24363:'pib_mensal'})

In [23]:
df[df["codigo"] == "ipca"]

,codigo,data,valor
0,ipca,01/05/2004,0.51
1,ipca,01/06/2004,0.71
2,ipca,01/07/2004,0.91
3,ipca,01/08/2004,0.69
4,ipca,01/09/2004,0.33
...,...,...,...
143,ipca,01/03/2026,0.88
144,ipca,01/04/2026,0.67
145,ipca,01/05/2026,0.58
146,ipca,01/06/2026,0.16


In [24]:
df["data"] = pd.to_datetime(df["data"], dayfirst=True)

date_spine = pd.DataFrame({
    "data": pd.date_range(
        start=df["data"].min(),
        end=df["data"].max(),
        freq="D"  # daily (use 'MS' to monthly)
    )
})
date_spine

,data
0,2004-05-01
1,2004-05-02
2,2004-05-03
3,2004-05-04
4,2004-05-05
...,...
8134,2026-08-08
8135,2026-08-09
8136,2026-08-10
8137,2026-08-11


In [25]:
df_wide = (
    df.pivot_table(
        index="data",
        columns="codigo",
        values="valor",
        aggfunc="last"
    )
    .reset_index()
)


In [26]:
df_wide = (
    date_spine
    .merge(df_wide, on="data", how="left")
    .sort_values("data")
)
df_wide


,data,ipca,pib_mensal,selic
0,2004-05-01,0.51,74.12622,NaN
1,2004-05-02,NaN,NaN,NaN
2,2004-05-03,NaN,NaN,NaN
3,2004-05-04,NaN,NaN,NaN
4,2004-05-05,NaN,NaN,NaN
...,...,...,...,...
8134,2026-08-08,NaN,NaN,NaN
8135,2026-08-09,NaN,NaN,NaN
8136,2026-08-10,NaN,NaN,0.05166
8137,2026-08-11,NaN,NaN,0.05166


In [27]:
# preenchendo todos os dias com os valores mensais/semestrais
df_wide["ipca"] = df_wide["ipca"].ffill()
df_wide["pib_mensal"] = df_wide["pib_mensal"].ffill()
df_wide["selic"] = df_wide["selic"].ffill() 

In [28]:
df_wide = df_wide[df_wide["data"] >= "2004-05-10"]
df_wide.index = df_wide["data"]

In [29]:
df_wide

,data,ipca,pib_mensal,selic
data,,,,
2004-05-10,2004-05-10,0.51,74.12622,0.058058
2004-05-11,2004-05-11,0.51,74.12622,0.058023
2004-05-12,2004-05-12,0.51,74.12622,0.057989
2004-05-13,2004-05-13,0.51,74.12622,0.058023
2004-05-14,2004-05-14,0.51,74.12622,0.058126
...,...,...,...,...
2026-08-08,2026-08-08,0.07,109.52993,0.051660
2026-08-09,2026-08-09,0.07,109.52993,0.051660
2026-08-10,2026-08-10,0.07,109.52993,0.051660


In [30]:
dolar = pd.read_parquet(r"..\data\dolar_enriched.parquet")

df_merged_dolar_selic = pd.merge(
    dolar,
    df_wide, 
    left_index=True,
    right_index=True,
    how="inner", 
    
    )

In [31]:
df_merged_dolar_selic

,Valor Venda Dolar,Valor Compra Dolar,Close_BZ=F,High_BZ=F,Low_BZ=F,Open_BZ=F,Volume_BZ=F,data,ipca,pib_mensal,selic
2007-07-30,1.8809,1.8801,75.739998,76.529999,75.440002,75.849998,2575,2007-07-30,0.24,89.38261,0.042956
2007-07-31,1.8776,1.8768,77.050003,77.169998,75.669998,75.699997,3513,2007-07-31,0.24,89.38261,0.042956
2007-08-01,1.8856,1.8848,75.349998,77.059998,74.860001,77.000000,3930,2007-08-01,0.47,90.22167,0.042956
2007-08-02,1.8729,1.8721,75.760002,76.209999,74.269997,75.220001,6180,2007-08-02,0.47,90.22167,0.042956
2007-08-03,1.8814,1.8806,74.750000,76.000000,74.529999,75.389999,4387,2007-08-03,0.47,90.22167,0.042956
...,...,...,...,...,...,...,...,...,...,...,...
2026-08-06,5.1017,5.1011,82.489998,83.769997,78.980003,79.430000,40651,2026-08-06,0.07,109.52993,0.051660
2026-08-07,5.0908,5.0902,83.550003,84.389999,81.500000,83.330002,37626,2026-08-07,0.07,109.52993,0.051660
2026-08-10,5.0963,5.0957,87.720001,87.919998,83.339996,83.559998,30023,2026-08-10,0.07,109.52993,0.051660
2026-08-11,5.1285,5.1279,88.910004,90.029999,86.730003,87.970001,30023,2026-08-11,0.07,109.52993,0.051660


In [32]:
df_merged_dolar_selic = df_merged_dolar_selic.drop(columns=["data"])

In [33]:
df_merged_dolar_selic.to_parquet(r"..\data\dolar_enriched.parquet")

In [34]:
df_merged_dolar_selic.columns

Index(['Valor Venda Dolar', 'Valor Compra Dolar', 'Close_BZ=F', 'High_BZ=F',
       'Low_BZ=F', 'Open_BZ=F', 'Volume_BZ=F', 'ipca', 'pib_mensal', 'selic'],
      dtype='object')

In [35]:
import pandas as pd
df = pd.read_parquet(r"..\data\dolar_enriched.parquet")

In [36]:
df[(df.index.year == 2019) & (df.index.month == 2)]#.index.value_counts()

,Valor Venda Dolar,Valor Compra Dolar,Close_BZ=F,High_BZ=F,Low_BZ=F,Open_BZ=F,Volume_BZ=F,ipca,pib_mensal,selic
2019-02-01,2.2192,2.2186,61.869999,61.869999,61.869999,61.869999,30044,0.46,102.87004,0.041063
2019-02-04,2.2192,2.2186,62.509998,63.630001,61.279999,62.700001,36025,0.46,102.87004,0.041063
2019-02-05,2.2192,2.2186,61.980000,63.020000,61.720001,62.779999,27285,0.46,102.87004,0.041063
2019-02-06,2.2192,2.2186,62.689999,62.799999,61.049999,62.040001,26542,0.46,102.87004,0.041063
2019-02-07,2.2192,2.2186,61.630001,62.900002,60.610001,62.660000,33982,0.46,102.87004,0.041063
2019-02-08,2.2192,2.2186,62.099998,62.349998,61.049999,61.599998,36111,0.46,102.87004,0.041063
2019-02-11,2.2192,2.2186,61.509998,62.340000,60.900002,62.000000,35285,0.46,102.87004,0.041063
2019-02-12,2.2192,2.2186,62.419998,63.320000,61.540001,61.599998,36801,0.46,102.87004,0.041063
2019-02-13,2.2192,2.2186,63.610001,63.959999,62.650002,62.650002,33859,0.46,102.87004,0.041063
2019-02-14,2.2192,2.2186,64.570000,64.809998,63.279999,63.630001,39526,0.46,102.87004,0.041063
